# Student Task: Prompt Chaining for Story Writing

## Goal
In this task, you will build a short science-fiction story using prompt chaining and iterative generation.

You will guide the model through these stages:

```text
Premise -> Outline -> Opening -> Continuation -> Final story
```

Some code is already completed for you. Complete every section marked `TODO`.

## Learning objectives

By the end of this task, you should be able to:

- Explain how prompt chaining passes output from one prompt into the next.
- Use a persona, context, constraints, and output instructions in a prompt.
- Continue a long generation over multiple model calls.
- Add a stopping condition to a generation loop.
- Count and inspect the final output.

## Task requirements

Your final program should:

1. Generate a one-sentence premise about a lost city on Mars.
2. Generate a story outline using the premise.
3. Generate the opening section using the premise and outline.
4. Continue the story at least twice.
5. Stop when the model writes `IAMDONE` or when the maximum number of continuations is reached.
6. Print the final story and its word count.

Do not place your API key directly in the notebook.

## 1. Setup

In [1]:
# !pip install -q -U openai

from getpass import getpass
from openai import OpenAI

api_key = getpass("Enter your OpenAI API key: ")
client = OpenAI(api_key=api_key)
MODEL = "gpt-4o-mini"

print("Setup complete")

Setup complete


## 2. Shared writing instructions

The variables below are completed. You may improve the writing guidelines after the main task works.

In [2]:
persona = "You are a creative science-fiction author writing for a general audience."

guidelines = """
Write vivid scenes with sensory details.
Develop the characters' goals and conflicts.
Do not summarize the story too quickly.
Continue naturally from the existing draft.
"""

print(persona)

You are a creative science-fiction author writing for a general audience.


## 3. Build the prompt chain

The first prompt is provided as an example. Complete the outline, opening, and continuation prompts.

Remember: `{{premise}}`, `{{outline}}`, and `{{story_text}}` are placeholders. They will be filled later using `.format()`.

In [3]:
premise_prompt = f"""
{persona}

Write one exciting sentence for a science-fiction story about a lost city on Mars.
"""

# Write outline_prompt using the premise placeholder.
# Ask the model for 5-7 major plot points.
outline_prompt = f"""\
{persona}

You have a gripping premise in mind:

{{premise}}

Write an outline for this story as 5-7 major plot points.
Number each point. One or two sentences each.
Do not write the story itself yet."""

# Write starting_prompt using the premise and outline placeholders.
# Ask for 500-800 words and introduce at least one important character.
starting_prompt = f"""\
{persona}

You have a gripping premise in mind:

{{premise}}

Your outline:

{{outline}}

Write the opening section of the story, covering only the first plot point.
Introduce at least one important character by name and make their goal clear.
Write between 500 and 800 words. Do not finish the story.

{guidelines}"""

# Write continuation_prompt using premise, outline, and story_text.
# Ask the model to continue the story and write IAMDONE when completely finished.
continuation_prompt = f"""\
{persona}

You have a gripping premise in mind:

{{premise}}

Your outline:

{{outline}}

The story so far:

{{story_text}}

=====

Continue the story from exactly where it stops. Cover only the next part of
the outline. Do not repeat or summarize what is already written.
Do not conclude the story in this section.

The outline has several numbered plot points. Write the word IAMDONE on its own
line only after the final numbered plot point has been fully written. If any
plot point is still uncovered, do not write IAMDONE.

{guidelines}"""

## 4. Generate the premise

The first model call creates the one-sentence story premise.

In [4]:
# Generate the outline with outline_prompt.format(premise=premise).
# Save the model output in outline and print it.
response = client.responses.create(
    model=MODEL,
    input=premise_prompt,
)

premise = response.output_text.strip()

print("Premise:")
print(premise)

Premise:
Beneath the rust-colored sands of Mars, a flicker of neon light revealed the entrance to a lost city, its towering spires humming with ancient technology and secrets that could rewrite humanity's understanding of its own history.


## 5. Generate the outline

The outline prompt receives the generated premise and asks the model to plan the major plot points.

In [5]:
response = client.responses.create(
    model=MODEL,
    input=outline_prompt.format(premise=premise),
)

outline = response.output_text.strip()
print("Outline:")
print(outline)

Outline:
### Outline for "City Beneath the Sands"

1. **Discovery of the Anomaly**  
A routine Mars exploration mission by the multinational crew aboard the *Ares One* detects an unusual electromagnetic pulse emanating from beneath the surface, prompting a detour to investigate.

2. **Unearthing the City**  
The team’s excavation reveals a vast underground city, its neon luminescence illuminating the intricate architecture and dormant technology, sparking an immediate mix of awe and caution among the crew.

3. **Awakening the Guardians**  
As the crew interacts with the ancient technology, they inadvertently reactivate automated defense systems. Mysterious holographic entities emerge, protecting the city’s secrets and testing the crew's intentions.

4. **Historical Revelation**  
While navigating through the city, the crew finds murals and inscriptions that reveal the civilization's history and their connection to Earth, suggesting an ancient interplanetary society that predates human 

## 6. Generate the opening

The opening prompt receives both the premise and the outline, then asks the model to begin the story.

In [6]:
# Generate the opening with starting_prompt.format(...).
# Save the model output in starting_draft and print it.
response = client.responses.create(
    model=MODEL,
    input=starting_prompt.format(
        premise=premise,
        outline=outline,
    ),
)

starting_draft = response.output_text.strip()

print("Opening:")
print(starting_draft)

Opening:
### City Beneath the Sands

The Martian landscape stretched out like an ocean of rust, undulating beneath the weight of a sun obscured by thin clouds. The *Ares One* hummed quietly, its metallic body a sliver of humanity’s ambition pitted against the infinite vastness. Inside the cramped confines of the ship, instruments blinked and chirped, their robotic voices filling the air with an electric pulse of anticipation.

Dr. Sofia Ramirez sat in the cramped control room, her fingers dancing across the illuminated console. With each keystroke, she felt the rhythm of her heart quicken, the promise of discovery weighing heavily on her shoulders. As the lead astrobiologist on the mission, she had spent years studying Mars from afar, envisioning its secrets. Now, out of the blue, there was a chance to uncover something unimaginable.

“Dr. Ramirez, we have a spike in electromagnetic activity at coordinates 43.12° N, 56.72° W,” a voice crackled through the comms, breaking the tension th

## 7. Continue the story once

The first continuation is generated manually so the process can be inspected before the iterative loop runs.

In [7]:
draft = starting_draft

# 6: Call the model with continuation_prompt.format(...).
# Use premise, outline, and story_text=draft.
response = client.responses.create(
    model=MODEL,
    input=continuation_prompt.format(
        premise=premise,
        outline=outline,
        story_text=draft,
    ),
)

continuation = response.output_text.strip()

print("Continuation:")
print(continuation)

Continuation:
### City Beneath the Sands - Continued

As they stepped into the cavern, the air thickened with an almost electric energy, tinged with the scent of distant metal and something faintly sweet. Shadows flickered on the walls, cast by their flashlights as they illuminated the grand architecture towering above them. Sofia’s heart raced—this was more than just an ancient city; it was a monument of civilization, every inch shouting of forgotten stories and ages past. 

“Just look at those murals,” Clara whispered, her voice barely above a breath. She moved toward the nearest wall, tracing delicate fingers along the intricate carvings. Her eyes widened as she deciphered images of beings—not human—interacting with what seemed to be astrological phenomena.

Sofia joined her, peering closely at the vibrant colors still vivid despite the centuries that had passed. “These… they could be the key to understanding this civilization’s purpose.” Each symbol seemed to resonate with her ferv

## 8. Build the iterative loop

The loop appends each continuation to the draft and stops when the model writes `IAMDONE` or the maximum number of calls is reached.

In [8]:
# 7: Add the first continuation to draft.
draft = draft + "\n\n" + continuation

# 8: Set a maximum number of additional calls.
MAX_CONTINUATIONS = 3

for _ in range(MAX_CONTINUATIONS):
    # 9: Stop if IAMDONE appears in continuation.
    if "IAMDONE" in continuation:
        break

    # 10: Request the next continuation.
    response = client.responses.create(
        model=MODEL,
        input=continuation_prompt.format(
            premise=premise,
            outline=outline,
            story_text=draft,
        ),
    )

    continuation = response.output_text.strip()

    # 11: Append the new continuation to draft.
    draft = draft + "\n\n" + continuation

# 12: Remove IAMDONE and save the cleaned story in final.
final = draft.replace("IAMDONE", "").strip()
print(final)

### City Beneath the Sands

The Martian landscape stretched out like an ocean of rust, undulating beneath the weight of a sun obscured by thin clouds. The *Ares One* hummed quietly, its metallic body a sliver of humanity’s ambition pitted against the infinite vastness. Inside the cramped confines of the ship, instruments blinked and chirped, their robotic voices filling the air with an electric pulse of anticipation.

Dr. Sofia Ramirez sat in the cramped control room, her fingers dancing across the illuminated console. With each keystroke, she felt the rhythm of her heart quicken, the promise of discovery weighing heavily on her shoulders. As the lead astrobiologist on the mission, she had spent years studying Mars from afar, envisioning its secrets. Now, out of the blue, there was a chance to uncover something unimaginable.

“Dr. Ramirez, we have a spike in electromagnetic activity at coordinates 43.12° N, 56.72° W,” a voice crackled through the comms, breaking the tension that had en

## 9. Evaluate the result

The final output is printed, its word count is calculated, and the completion marker is checked.

In [9]:
# 13: Count the words in final.
word_count = len(final.split())
print(f"Final story word count: {word_count}")

# Extra check added for this student task: verify that the completion marker is removed.
print("Completion marker removed:", "IAMDONE" not in final)

Final story word count: 1949
Completion marker removed: True


## Submission checklist

- [x] All `TODO` sections are completed.
- [x] The notebook runs from top to bottom without errors.
- [x] The premise, outline, opening, and final story are printed.
- [ ] The final story contains at least two continuations.
- [x] The word count is printed.
- [ ] Reflection questions are answered.
- [x] No API key is written directly in the notebook.